# BirdCLEF+ 2026 - Insightful EDA

**Purpose.** Build a clear picture of the BirdCLEF+ 2026 training data before modeling: label imbalance, taxonomy coverage, duration/chunking behavior, secondary-label noise, soundscape annotations, and representative audio examples.

**Run mode.** Kaggle analysis notebook; no model training.

**Primary outputs.** CSV summaries, diagnostic plots, representative mel-spectrograms, and a zipped artifact bundle under `/kaggle/working`.

Artifacts are written to `/kaggle/working/artifacts/eda`.


# 1. Setup And Inputs


## 1.1 Environment Setup


In [ ]:
from pathlib import Path
import json
import os
import random
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)


class CFG:
    seed = 42
    competition_name = "birdclef-2026"
    data_root = None
    artifact_dir = Path("/kaggle/working/artifacts")


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)


def find_data_root() -> Path:
    candidates = [
        Path("/kaggle/input/birdclef-2026"),
        Path("/kaggle/input/birdclef-2026-repack/birdclef-2026"),
        Path("/kaggle/input/birdclef-2026-repack"),
        Path("data/raw/birdclef-2026"),
    ]
    for path in candidates:
        if (path / "train.csv").exists():
            return path
    input_root = Path("/kaggle/input")
    if input_root.exists():
        matches = list(input_root.glob("**/train.csv"))
        if matches:
            return matches[0].parent
    raise FileNotFoundError("Could not find train.csv. Attach the BirdCLEF 2026 dataset.")


def read_optional_csv(path: Path) -> pd.DataFrame | None:
    return pd.read_csv(path) if path.exists() else None


seed_everything(CFG.seed)
CFG.data_root = find_data_root()
CFG.artifact_dir.mkdir(parents=True, exist_ok=True)

print(f"Data root: {CFG.data_root}")
print(f"Artifacts: {CFG.artifact_dir}")

import ast
from collections import Counter

import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Audio, display

CFG.artifact_dir = CFG.artifact_dir / "eda"
CFG.artifact_dir.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")


class CFG(CFG):
    sample_rate = 32000
    clip_seconds = 5
    n_mels = 128
    top_n = 30
    random_examples = 6


## 1.2 Load Metadata


**Insight goal.** Before modeling, confirm the competition files are mounted correctly and that the training table, taxonomy table, soundscape labels, and sample submission agree with each other. This section creates the basic dataset inventory that later sections use for imbalance, duration, and domain-shift diagnostics.


In [ ]:
def parse_label_list(value) -> list[str]:
    if isinstance(value, list):
        return [str(x) for x in value]
    if pd.isna(value):
        return []
    try:
        parsed = ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []
    return [str(x) for x in parsed] if isinstance(parsed, list) else []


train = pd.read_csv(CFG.data_root / "train.csv")
taxonomy = read_optional_csv(CFG.data_root / "taxonomy.csv")
soundscape_labels = read_optional_csv(CFG.data_root / "train_soundscapes_labels.csv")
sample_submission = read_optional_csv(CFG.data_root / "sample_submission.csv")

train["filepath"] = train["filename"].map(lambda x: CFG.data_root / "train_audio" / x)
if "secondary_labels" in train.columns:
    train["secondary_labels"] = train["secondary_labels"].map(parse_label_list)
else:
    train["secondary_labels"] = [[] for _ in range(len(train))]

if taxonomy is not None and "primary_label" in taxonomy.columns:
    train = train.merge(taxonomy, on="primary_label", how="left", suffixes=("", "_taxonomy"))

summary = {
    "train_rows": len(train),
    "primary_classes": train["primary_label"].nunique(),
    "taxonomy_rows": 0 if taxonomy is None else len(taxonomy),
    "soundscape_label_rows": 0 if soundscape_labels is None else len(soundscape_labels),
    "sample_submission_rows": 0 if sample_submission is None else len(sample_submission),
    "audio_files_found": int(train["filepath"].map(Path.exists).sum()),
}
pd.Series(summary).to_csv(CFG.artifact_dir / "dataset_summary.csv", header=["value"])
display(pd.Series(summary).to_frame("value"))
display(train.head())


# 2. Data Integrity Audit


## 2.1 Dataset Schema And Missingness


**What to look for.** Missing metadata can create silent leakage or brittle preprocessing logic. The most important fields for the baseline notebooks are `filename`, `primary_label`, `secondary_labels`, and `duration`; if any of those are incomplete, the training pipeline needs explicit fallback behavior.


In [ ]:
def safe_nunique(series: pd.Series) -> int:
    values = series.map(make_hashable)
    return int(values.nunique(dropna=True))


def make_hashable(value):
    if isinstance(value, list):
        return tuple(make_hashable(item) for item in value)
    if isinstance(value, np.ndarray):
        return tuple(make_hashable(item) for item in value.tolist())
    if isinstance(value, dict):
        return tuple(sorted(((key, make_hashable(item)) for key, item in value.items()), key=repr))
    if isinstance(value, set):
        return tuple(sorted((make_hashable(item) for item in value), key=repr))
    return value


schema = pd.DataFrame(
    {
        "column": train.columns,
        "dtype": [str(train[col].dtype) for col in train.columns],
        "missing": [int(train[col].isna().sum()) for col in train.columns],
        "missing_pct": [float(train[col].isna().mean()) for col in train.columns],
        "unique": [safe_nunique(train[col]) for col in train.columns],
    }
)
schema.to_csv(CFG.artifact_dir / "train_schema_missingness.csv", index=False)
display(schema)

missing_files = train.loc[~train["filepath"].map(Path.exists), ["filename", "filepath"]]
missing_files.to_csv(CFG.artifact_dir / "missing_audio_files.csv", index=False)
print(f"Missing audio files: {len(missing_files):,}")


## 2.2 Duplicate Records


**Why this matters.** Duplicate metadata rows can inflate apparent sample size and leak repeated labels into validation. The soundscape label file is especially important because exact duplicate 5-second annotations can bias multi-label prevalence estimates.


In [ ]:
duplicate_tables = {}
def duplicate_ready(frame: pd.DataFrame) -> pd.DataFrame:
    return frame.apply(lambda col: col.map(make_hashable) if col.dtype == "object" else col)


for name, frame in {
    "train": train,
    "taxonomy": taxonomy,
    "soundscape_labels": soundscape_labels,
    "sample_submission": sample_submission,
}.items():
    if frame is None:
        continue
    frame_for_dupes = duplicate_ready(frame)
    dup_mask = frame_for_dupes.duplicated(keep=False)
    duplicate_tables[name] = int(dup_mask.sum())
    frame.loc[dup_mask].to_csv(CFG.artifact_dir / f"{name}_duplicate_rows.csv", index=False)

duplicate_summary = pd.Series(duplicate_tables, name="duplicate_rows").rename_axis("table").reset_index()
duplicate_summary.to_csv(CFG.artifact_dir / "duplicate_summary.csv", index=False)
display(duplicate_summary)

key_checks = []
for name, frame, candidates in [
    ("train", train, [["filename"], ["filepath"], ["primary_label", "filename"]]),
    ("taxonomy", taxonomy, [["primary_label"], ["scientific_name"]]),
    ("soundscape_labels", soundscape_labels, [["row_id"], ["filename"], ["filename", "primary_label"]]),
    ("sample_submission", sample_submission, [["row_id"]]),
]:
    if frame is None:
        continue
    frame_for_dupes = duplicate_ready(frame)
    for keys in candidates:
        if all(key in frame.columns for key in keys):
            key_dup_mask = frame_for_dupes.duplicated(subset=keys, keep=False)
            dup_count = int(key_dup_mask.sum())
            key_checks.append({"table": name, "keys": "+".join(keys), "duplicate_rows": dup_count})
            if dup_count:
                frame.loc[key_dup_mask].sort_values(keys).to_csv(
                    CFG.artifact_dir / f"{name}_{'_'.join(keys)}_duplicates.csv",
                    index=False,
                )

key_duplicate_summary = pd.DataFrame(key_checks)
key_duplicate_summary.to_csv(CFG.artifact_dir / "key_duplicate_summary.csv", index=False)
display(key_duplicate_summary)

if soundscape_labels is not None:
    soundscape_dedup = soundscape_labels.loc[~duplicate_ready(soundscape_labels).duplicated()].reset_index(drop=True)
    print(f"Soundscape labels before deduplication: {len(soundscape_labels):,}")
    print(f"Soundscape labels after deduplication: {len(soundscape_dedup):,}")
else:
    soundscape_dedup = None


**Takeaway.** If duplicate rows appear in `train_soundscapes_labels.csv`, downstream soundscape prevalence and co-occurrence analysis should use the deduplicated table. The original file is still copied to artifacts for auditability.


# 3. Training Label Analysis


## 3.1 Primary Label Imbalance


**Why this matters.** BirdCLEF-style datasets are usually long-tailed: common species dominate the loss, while rare species are easy to ignore. The imbalance plots below help decide whether the first baseline should use balanced sampling, class-aware augmentation, focal loss, or per-class validation diagnostics.


In [ ]:
label_counts = (
    train["primary_label"]
    .value_counts()
    .rename_axis("primary_label")
    .reset_index(name="recordings")
)
label_counts["share"] = label_counts["recordings"] / label_counts["recordings"].sum()
label_counts["cumulative_share"] = label_counts["share"].cumsum()
label_counts["imbalance_ratio_vs_median"] = label_counts["recordings"] / label_counts["recordings"].median()
label_counts.to_csv(CFG.artifact_dir / "primary_label_counts.csv", index=False)

head_share = label_counts.head(10)["share"].sum()
tail_singletons = int((label_counts["recordings"] == 1).sum())
print(f"Top 10 classes contain {head_share:.1%} of recordings.")
print(f"Singleton classes: {tail_singletons:,}")
display(label_counts.head(CFG.top_n))
display(label_counts.tail(CFG.top_n))

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=label_counts.head(CFG.top_n), x="recordings", y="primary_label", ax=ax, color="#3C78D8")
ax.set_title(f"Top {CFG.top_n} primary labels by recording count")
ax.set_xlabel("recordings")
ax.set_ylabel("")
fig.tight_layout()
fig.savefig(CFG.artifact_dir / "top_primary_labels.png", dpi=160)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(label_counts["recordings"], bins=50, ax=axes[0], color="#2CA02C")
axes[0].set_title("Class-count distribution")
axes[0].set_xlabel("recordings per class")
sns.lineplot(data=label_counts.reset_index(), x="index", y="cumulative_share", ax=axes[1], color="#D62728")
axes[1].set_title("Cumulative share by ranked class")
axes[1].set_xlabel("class rank")
axes[1].set_ylabel("cumulative share")
fig.tight_layout()
fig.savefig(CFG.artifact_dir / "class_imbalance_diagnostics.png", dpi=160)
plt.show()


## 3.2 Duration And Chunking Implications


**Why this matters.** A 5-second crop is a modeling choice, not just a preprocessing detail. Short clips may need padding, long recordings can provide many training crops, and classes with fewer files but longer total duration may be less data-poor than raw recording counts suggest.


In [ ]:
if "duration" in train.columns:
    duration_summary = train["duration"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
    duration_summary.to_csv(CFG.artifact_dir / "duration_summary.csv", header=["value"])
    display(duration_summary)

    duration_by_label = (
        train.groupby("primary_label")["duration"]
        .agg(recordings="count", median_duration="median", total_seconds="sum")
        .reset_index()
        .sort_values("total_seconds", ascending=False)
    )
    duration_by_label["estimated_5s_chunks"] = np.ceil(duration_by_label["total_seconds"] / CFG.clip_seconds).astype(int)
    duration_by_label.to_csv(CFG.artifact_dir / "duration_by_primary_label.csv", index=False)
    display(duration_by_label.head(20))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    clipped = train["duration"].clip(upper=train["duration"].quantile(0.99))
    sns.histplot(clipped, bins=60, ax=axes[0], color="#9467BD")
    axes[0].set_title("Audio duration distribution, clipped at p99")
    axes[0].set_xlabel("seconds")
    sns.scatterplot(
        data=duration_by_label,
        x="recordings",
        y="total_seconds",
        size="median_duration",
        sizes=(20, 180),
        alpha=0.7,
        ax=axes[1],
        color="#FF7F0E",
    )
    axes[1].set_xscale("log")
    axes[1].set_yscale("log")
    axes[1].set_title("Per-class recording count vs total audio")
    axes[1].set_xlabel("recordings, log scale")
    axes[1].set_ylabel("total seconds, log scale")
    fig.tight_layout()
    fig.savefig(CFG.artifact_dir / "duration_and_chunking.png", dpi=160)
    plt.show()
else:
    print("No duration column found in train.csv.")


## 3.3 Secondary Labels And Co-Occurrence


**Why this matters.** Secondary labels are noisy but valuable. They reveal species that often appear together and can later support soft labels, multi-label training, mixup targets, or post-processing rules. For the first EfficientNet baseline, they are kept diagnostic rather than target-defining.


In [ ]:
secondary = train[["filename", "primary_label", "secondary_labels"]].explode("secondary_labels")
secondary = secondary.dropna(subset=["secondary_labels"])
secondary_counts = (
    secondary["secondary_labels"]
    .value_counts()
    .rename_axis("secondary_label")
    .reset_index(name="mentions")
)
secondary_counts.to_csv(CFG.artifact_dir / "secondary_label_counts.csv", index=False)
rows_with_secondary = int((train["secondary_labels"].map(len) > 0).sum())
print(f"Rows with secondary labels: {rows_with_secondary:,} ({rows_with_secondary / len(train):.1%})")
display(secondary_counts.head(CFG.top_n))

cooccurrence = (
    secondary.groupby(["primary_label", "secondary_labels"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
cooccurrence.to_csv(CFG.artifact_dir / "primary_secondary_cooccurrence.csv", index=False)
display(cooccurrence.head(30))

if len(secondary_counts):
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.barplot(data=secondary_counts.head(CFG.top_n), x="mentions", y="secondary_label", ax=ax, color="#17BECF")
    ax.set_title(f"Top {CFG.top_n} secondary labels")
    ax.set_xlabel("mentions")
    ax.set_ylabel("")
    fig.tight_layout()
    fig.savefig(CFG.artifact_dir / "top_secondary_labels.png", dpi=160)
    plt.show()


## 3.4 Taxonomy Coverage


**Why this matters.** Taxonomy metadata gives a way to audit errors above the species level. If the model confuses species within the same genus or family, that is a different failure mode from confusing unrelated calls. This table also checks whether train labels and taxonomy labels are aligned.


In [ ]:
if taxonomy is not None:
    taxonomy.to_csv(CFG.artifact_dir / "taxonomy_copy.csv", index=False)
    display(taxonomy.head())
    taxonomy_cols = taxonomy.columns.tolist()
    print(taxonomy_cols)

    taxonomy_label_col = "primary_label" if "primary_label" in taxonomy.columns else None
    if taxonomy_label_col:
        train_labels = set(pd.read_csv(CFG.data_root / "train.csv")["primary_label"].unique())
        taxonomy_labels = set(taxonomy[taxonomy_label_col].dropna().astype(str).unique())
        coverage = {
            "train_labels": len(train_labels),
            "taxonomy_labels": len(taxonomy_labels),
            "train_labels_missing_from_taxonomy": len(train_labels - taxonomy_labels),
            "taxonomy_labels_not_in_train": len(taxonomy_labels - train_labels),
        }
        pd.Series(coverage).to_csv(CFG.artifact_dir / "taxonomy_coverage.csv", header=["value"])
        display(pd.Series(coverage).to_frame("value"))

    candidate_cols = [c for c in ["class_name", "order", "family", "genus", "species", "common_name", "scientific_name"] if c in train.columns]
    for col in candidate_cols[:4]:
        counts = train[col].value_counts(dropna=False).head(20).rename_axis(col).reset_index(name="recordings")
        display(counts)
else:
    print("No taxonomy.csv found.")


# 4. Metadata And Domain Shift


## 4.1 Metadata Quality And Geography


**Why this matters.** Metadata quality is uneven across recording sources. Rating, collection, author, license, and geography can all become hidden confounders: a model may learn recorder/source artifacts instead of species-specific acoustic structure.


In [ ]:
metadata_outputs = {}

if "rating" in train.columns:
    rating_counts = train["rating"].value_counts(dropna=False).sort_index().rename_axis("rating").reset_index(name="recordings")
    rating_counts.to_csv(CFG.artifact_dir / "rating_counts.csv", index=False)
    display(rating_counts)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.barplot(data=rating_counts, x="rating", y="recordings", ax=axes[0], color="#7F7F7F")
    axes[0].set_title("Recording rating distribution")
    if "class_name" in train.columns:
        quality_by_class = train.groupby("class_name")["rating"].agg(["count", "mean", "median"]).reset_index()
        quality_by_class.to_csv(CFG.artifact_dir / "quality_by_class.csv", index=False)
        sns.boxplot(data=train, x="rating", y="class_name", ax=axes[1], color="#BCBD22")
        axes[1].set_title("Rating by biological class")
    else:
        axes[1].axis("off")
    fig.tight_layout()
    fig.savefig(CFG.artifact_dir / "metadata_quality_rating.png", dpi=160)
    plt.show()

for col in ["collection", "license", "type", "author"]:
    if col in train.columns:
        counts = train[col].value_counts(dropna=False).head(30).rename_axis(col).reset_index(name="recordings")
        counts.to_csv(CFG.artifact_dir / f"{col}_counts.csv", index=False)
        metadata_outputs[col] = len(counts)

if {"latitude", "longitude"}.issubset(train.columns):
    pantanal = {"lat_min": -21.6, "lat_max": -16.5, "lon_min": -57.6, "lon_max": -55.9}
    geo = train.dropna(subset=["latitude", "longitude"]).copy()
    geo["inside_pantanal_box"] = (
        geo["latitude"].between(pantanal["lat_min"], pantanal["lat_max"])
        & geo["longitude"].between(pantanal["lon_min"], pantanal["lon_max"])
    )
    geo_summary = pd.Series(
        {
            "records_with_coordinates": len(geo),
            "inside_pantanal_box": int(geo["inside_pantanal_box"].sum()),
            "inside_pantanal_share": float(geo["inside_pantanal_box"].mean()),
            "species_inside_pantanal": int(geo.loc[geo["inside_pantanal_box"], "primary_label"].nunique()),
        }
    )
    geo_summary.to_csv(CFG.artifact_dir / "geography_summary.csv", header=["value"])
    display(geo_summary.to_frame("value"))

    fig, ax = plt.subplots(figsize=(9, 6))
    sns.scatterplot(
        data=geo.sample(min(len(geo), 12000), random_state=CFG.seed),
        x="longitude",
        y="latitude",
        hue="inside_pantanal_box",
        s=8,
        alpha=0.35,
        ax=ax,
    )
    ax.set_title("Training recording geography")
    fig.tight_layout()
    fig.savefig(CFG.artifact_dir / "recording_geography.png", dpi=160)
    plt.show()


**Takeaway.** Treat quality/source/geography as potential domain-shift variables. Validation splits grouped by author, recording, or source can be more honest than purely random splits, especially when the hidden soundscapes come from a narrower ecological region.


## 4.2 Soundscape Labels


**Why this matters.** Soundscapes are closer to the evaluation domain than clean training clips: longer recordings, overlapping calls, background noise, and sparse temporal annotations. Treating this table as a separate domain helps avoid over-trusting validation metrics from clean clips only.


In [ ]:
if soundscape_labels is None:
    print("No train_soundscapes_labels.csv found.")
else:
    soundscape_labels.to_csv(CFG.artifact_dir / "soundscape_labels_copy.csv", index=False)
    soundscape_analysis = soundscape_dedup if "soundscape_dedup" in globals() and soundscape_dedup is not None else soundscape_labels
    display(soundscape_analysis.head())
    print(soundscape_analysis.columns.tolist())

    label_like_cols = [c for c in soundscape_analysis.columns if "label" in c.lower() or "species" in c.lower() or "code" in c.lower()]
    time_like_cols = [c for c in soundscape_analysis.columns if "time" in c.lower() or "second" in c.lower()]
    file_like_cols = [c for c in soundscape_analysis.columns if "filename" in c.lower() or "soundscape" in c.lower() or "row_id" in c.lower()]

    soundscape_summary = {
        "rows": len(soundscape_analysis),
        "columns": len(soundscape_analysis.columns),
        "label_like_columns": ", ".join(label_like_cols),
        "time_like_columns": ", ".join(time_like_cols),
        "file_like_columns": ", ".join(file_like_cols),
    }
    pd.Series(soundscape_summary).to_csv(CFG.artifact_dir / "soundscape_summary.csv", header=["value"])
    display(pd.Series(soundscape_summary).to_frame("value"))

    if label_like_cols:
        col = label_like_cols[0]
        sc_counts = soundscape_analysis[col].value_counts().head(CFG.top_n).rename_axis(col).reset_index(name="rows")
        sc_counts.to_csv(CFG.artifact_dir / "soundscape_label_counts.csv", index=False)
        display(sc_counts)

    if {"filename", "primary_label"}.issubset(soundscape_analysis.columns):
        sc = soundscape_analysis.copy()
        sc["label_list"] = sc["primary_label"].astype(str).str.split(";")
        sc["n_labels"] = sc["label_list"].map(len)
        sc["site"] = sc["filename"].astype(str).str.extract(r"_(S\d+)_", expand=False)
        sc["hour"] = sc["filename"].astype(str).str.extract(r"_(\d{6})\.ogg$", expand=False).str[:2]
        overlap_summary = sc["n_labels"].describe(percentiles=[0.5, 0.75, 0.9]).to_frame("value")
        overlap_summary.to_csv(CFG.artifact_dir / "soundscape_overlap_summary.csv")
        display(overlap_summary)

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        sns.histplot(sc["n_labels"], discrete=True, ax=axes[0], color="#8C564B")
        axes[0].set_title("Labels per soundscape segment")
        if sc["hour"].notna().any():
            hour_counts = sc["hour"].value_counts().sort_index().rename_axis("hour").reset_index(name="segments")
            hour_counts.to_csv(CFG.artifact_dir / "soundscape_hour_counts.csv", index=False)
            sns.barplot(data=hour_counts, x="hour", y="segments", ax=axes[1], color="#E377C2")
            axes[1].set_title("Labeled soundscape segments by hour")
        else:
            axes[1].axis("off")
        fig.tight_layout()
        fig.savefig(CFG.artifact_dir / "soundscape_overlap_time.png", dpi=160)
        plt.show()


**Takeaway.** Soundscape labels should be interpreted after deduplication and with time/site context. Multi-label density and temporal clustering are clues for hour-aware priors, threshold tuning, and validation design.


## 4.3 Representative Audio And Spectrograms


**Why this matters.** A few spectrograms often reveal issues that tables hide: silence, clipping, background insects, rain, distant calls, and frequency bands that matter for augmentation. These examples are not a validation set; they are a quick sanity check for the acoustic texture the model will see.


In [ ]:
def load_clip(path: Path, seconds: float = 5.0) -> np.ndarray:
    y, _ = librosa.load(path, sr=CFG.sample_rate, mono=True, duration=seconds)
    target_len = int(CFG.sample_rate * seconds)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    return y[:target_len].astype(np.float32)


available = train[train["filepath"].map(Path.exists)].copy()
if len(available) == 0:
    print("No local audio files available for previews.")
else:
    example_df = (
        available.groupby("primary_label", group_keys=False)
        .apply(lambda x: x.sample(1, random_state=CFG.seed))
        .sample(min(CFG.random_examples, available["primary_label"].nunique()), random_state=CFG.seed)
        .reset_index(drop=True)
    )
    example_df[["filename", "primary_label"]].to_csv(CFG.artifact_dir / "audio_examples.csv", index=False)
    display(example_df[["filename", "primary_label"]])

    fig, axes = plt.subplots(len(example_df), 1, figsize=(12, 2.6 * len(example_df)))
    axes = np.atleast_1d(axes)
    for ax, (_, row) in zip(axes, example_df.iterrows()):
        y = load_clip(row["filepath"], CFG.clip_seconds)
        mel = librosa.feature.melspectrogram(y=y, sr=CFG.sample_rate, n_mels=CFG.n_mels)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        librosa.display.specshow(mel_db, sr=CFG.sample_rate, x_axis="time", y_axis="mel", ax=ax)
        ax.set_title(f"{row['primary_label']} | {row['filename']}")
    fig.tight_layout()
    fig.savefig(CFG.artifact_dir / "representative_mels.png", dpi=160)
    plt.show()

    first = example_df.iloc[0]
    print(f"Audio preview: {first['primary_label']} | {first['filename']}")
    display(Audio(load_clip(first["filepath"], CFG.clip_seconds), rate=CFG.sample_rate))


# 5. Modeling Decisions And Artifacts


## 5.1 Analysis Takeaways


The diagnostics above point to a few concrete modeling decisions:

1. **Class imbalance is the first-order risk.** The ranked-class curve rises quickly, and our saved EDA artifacts show that the top 30 labels account for roughly 40% of recordings while dozens of classes have very little data. A plain random sampler will over-optimize for head classes unless we add class-aware sampling, rare-class augmentation, or per-class monitoring.

2. **Recording count is not the same as acoustic coverage.** Duration diagnostics help identify classes that have fewer files but more total seconds, and classes that have many short clips. A fixed 5-second crop is a reasonable baseline, but long clips should eventually support random crops during training and multi-crop averaging during inference.

3. **Secondary labels are useful signal, but not a clean first target.** The secondary-label plot shows repeated co-occurrence patterns. These labels are noisy enough that the first baseline should stay single-label, but they are valuable for later soft targets, mixup labels, and confusion analysis.

4. **Soundscapes are a domain shift, not just extra rows.** The soundscape label table contains multi-label combinations that look much closer to the evaluation setting than clean training clips. Validation based only on isolated `train_audio` clips can overstate leaderboard readiness.

5. **Spectrograms support both CNN and foundation-model strategies.** Representative mel plots show repeated phrases, sparse calls, background noise, and frequency-specific structure. That supports an EfficientNet mel baseline, while the stronger Perch probe result suggests Perch is best used as a teacher or feature source for a faster student model.


**How to use this section.** The notebook turns the diagnostics above into practical modeling implications. These takeaways should guide the baseline notebooks: validation design, crop length, augmentation strength, class balancing, and whether to compare clean-clip validation against soundscape-style artifacts.


In [ ]:
takeaways = []
takeaways.append(
    f"Primary-label imbalance is substantial: top 10 classes cover {label_counts.head(10)['share'].sum():.1%} of training recordings."
)
if "duration" in train.columns:
    takeaways.append(
        f"Median clip duration is {train['duration'].median():.1f}s; a {CFG.clip_seconds}s training window creates multiple possible crops for long recordings."
    )
takeaways.append(
    f"Secondary labels appear in {(train['secondary_labels'].map(len) > 0).mean():.1%} of rows, so noisy multi-label context may help later even if the first baseline is single-label."
)
if taxonomy is not None:
    takeaways.append("Taxonomy metadata can support stratified diagnostics and class-family level error analysis.")
if soundscape_labels is not None:
    takeaways.append("Soundscape annotations should be treated separately from clean training clips because they reflect the evaluation domain more closely.")

takeaways_df = pd.DataFrame({"takeaway": takeaways})
takeaways_df.to_csv(CFG.artifact_dir / "modeling_takeaways.csv", index=False)
display(takeaways_df)


## 5.2 Final Conclusion


This EDA suggests a practical modeling roadmap:

- Use the EfficientNet-B0 mel baseline as the dependable submission path because it is fast, pure PyTorch, and already competition-safe.
- Use Perch v2 as an offline teacher or feature generator rather than direct hidden-test inference, because the CUDA SavedModel is too slow and the CPU/cache path depends on external coverage.
- Improve validation before chasing architecture complexity: group leakage, source bias, geography drift, and soundscape overlap can all make local metrics misleading.
- Prioritize class-aware training, multi-crop inference, and eventually soundscape-informed calibration. These changes are more aligned with the observed data issues than simply scaling the backbone.


## 5.3 Artifact Manifest


In [ ]:
manifest = sorted(str(path.relative_to(CFG.artifact_dir)) for path in CFG.artifact_dir.glob("*"))
(CFG.artifact_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
manifest


## 5.4 Package Artifacts For Download


Kaggle makes files under `/kaggle/working` downloadable after the notebook finishes. This cell zips the EDA tables and figures into one file so you can download them, review them locally, and optionally commit selected lightweight artifacts such as `.csv`, `.json`, or `.png` files to GitHub.

Avoid committing large generated arrays or model checkpoints unless you intentionally manage them with Git LFS or a release asset.


In [ ]:
import zipfile
from IPython.display import FileLink

zip_path = Path("/kaggle/working/birdclef_eda_artifacts.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(CFG.artifact_dir.rglob("*")):
        if path.is_file():
            zf.write(path, arcname=path.relative_to(CFG.artifact_dir.parent))

print(f"Wrote {zip_path} ({zip_path.stat().st_size / 1024 / 1024:.2f} MB)")
display(FileLink(zip_path))
